# Predicción de Severidad de Crímenes · LAPD 2020-2024

**Dataset:** Crime Data from 2020 to 2024 — Los Angeles Police Department

**Objetivo:** Predecir si un crimen es de Tipo 1 (grave: homicidio, robo, agresión) o Tipo 2 (menor gravedad), y apoyar la toma de decisiones sobre asignación de recursos policiales.

**Autores:**
- Beiza Vargas Johar — [jobv99-ui ](#)
- Carrizosa Dávila Carlos Andrés - [carrytec-oss](#)
- Martinez García Fernando Gabriel - [gabo-01](#)



Resumen del archivo:

Archivo: Crime_Data_from_2020_to_2024.csv;
Tamaño: Archivo original en csv 1.2 GB, reducido en parquet a 244 MB;
Filas: 1,004,894 registros (sin encabezado);
Columnas 28:

| Columna | Descripción |
|---------|-------------|
|DR_NO.                       | Número de reporte |
|Date Rptd                    | Fecha de reporte  |
|DATE OCC	                 | Fecha del crimen  |
|TIME OCC	                 | Hora del crimen   |
|AREA / AREA NAME             | Área de LAPD.     |
|Crm Cd / Crm Cd Desc         | Código y descripción del crimen |
|Vict Age/Sex/Descent         | Datos de la víctima |
|Premis Desc	                 | Tipo de lugar |
|Weapon Used Cd / Weapon Desc | Arma usada |
|Status / Status Desc	     | Estado de la investigación |
|LAT / LON                    | Coordenadas geográficas |

---
## 1. Planteamiento del Problema

### Contexto de negocio
El LAPD gestiona 21 divisiones en Los Ángeles. La asignación eficiente de patrullas depende de anticipar qué tipo de crimen ocurrirá según zona, hora y características del entorno.

### Objetivo
Construir un modelo clasificador que prediga si un incidente será un **crimen grave (Tipo 1)** o **menos grave (Tipo 2)**, basándose en variables disponibles al momento del reporte.

### Variable objetivo
- `Tipo 1-2` → **1 = Crimen grave** (homicidio, robo, asalto con arma, violación)  
- `Tipo 1-2` → **2 = Crimen menor** (vandalismo, vagancia, fraude menor)

### Valor para la toma de decisiones
- Priorizar zonas de alto riesgo por turno horario
- Identificar patrones de escalamiento de crímenes
- Asignar recursos preventivos con mayor precisión

---
## 2. Preparación de Datos
### 2.1 Carga de datos

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

En esta sección cargamos el dataset desde Google Drive. El archivo está en formato Parquet, que es significativamente más rápido y ligero que el CSV original (244 MB → 41 MB). Antes de continuar, el código verifica que el archivo existe en la ruta correcta para evitar errores inesperados.

In [ ]:
from google.colab import drive
import pandas as pd
import os

# Montar Google Drive
drive.mount('/content/drive')

# Ruta dentro de tu Drive
RUTA_DATOS = "/content/drive/MyDrive/ML/Crime_Data.parquet"

# Verificar existencia
assert os.path.exists(RUTA_DATOS), f"Archivo no encontrado: {RUTA_DATOS}"
print(f"Archivo encontrado: {RUTA_DATOS}")

# Cargar datos
df = pd.read_parquet(RUTA_DATOS)

print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
df.head(3)

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")
df.head(3)

### 2.2 Limpieza de datos

Antes de analizar los datos es necesario limpiarlos. En este paso eliminamos registros con coordenadas inválidas (latitud y longitud en cero), rellenamos valores ausentes en variables como sexo y origen de la víctima, y creamos una variable indicadora de si se usó un arma. También eliminamos columnas con demasiados valores nulos o que no aportan valor al modelo.

In [ ]:
import numpy as np

raw_shape = df.shape

# Eliminar coordenadas inválidas (0,0)
df = df[(df['LAT'] != 0) & (df['LON'] != 0)]

# Rellenar valores nulos en variables categóricas
df['Vict Sex'] = df['Vict Sex'].fillna('X')
df['Vict Descent'] = df['Vict Descent'].fillna('X')
df['Premis Cd'] = df['Premis Cd'].fillna(0)

# Indicador de arma (1 si hubo arma, 0 si no)
df['has_weapon'] = df['Weapon Used Cd'].notna().astype(int)

# Columnas con demasiados nulos o no útiles para el modelo
cols_drop = ['Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'Cross Street',
             'Mocodes', 'Weapon Used Cd', 'Weapon Desc',
             'DR_NO', 'Rpt Dist No', 'Date Rptd', 'Status', 'LOCATION']
df.drop(columns=cols_drop, inplace=True)

# Variable objetivo binaria: 1=grave, 0=menor
df['target'] = (df['Part 1-2'] == 1).astype(int)

print(f"Forma original: {raw_shape}")
print(f"Forma limpia:   {df.shape}")
print(f"\nDistribución objetivo:\n{df['target'].value_counts(normalize=True).round(3)}")

### 2.3 Análisis Exploratorio (EDA)

El análisis exploratorio nos permite entender la estructura del dataset antes de modelar. Revisamos los tipos de datos de cada columna, la cantidad de registros válidos, y cuáles variables tienen valores ausentes que podrían afectar el entrenamiento del modelo.

In [ ]:
df.info()

In [ ]:
nulos = df.isnull().sum()
print("Nulos por columna:")
print(nulos[nulos > 0].sort_values(ascending=False) if nulos.any() else "Sin nulos")

Las siguientes gráficas resumen los patrones más relevantes del dataset: cómo se distribuyen los crímenes a lo largo del día, qué áreas concentran más crímenes graves, la relación entre uso de armas y severidad, y el perfil de edad de las víctimas. Estos patrones guían la selección de variables para el modelo.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA — Crime Data LAPD 2020-2024', fontsize=14, fontweight='bold')

# 1. Distribución objetivo
counts = df['target'].value_counts()
axes[0,0].bar(['Tipo 2\n(menor)', 'Tipo 1\n(grave)'],
              [counts[0], counts[1]], color=['steelblue','tomato'])
axes[0,0].set_title('Distribución Objetivo')
axes[0,0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'{x/1e3:.0f}K'))

# 2. Crímenes por hora
df['hour'] = df['TIME OCC'] // 100
hora_counts = df.groupby(['hour','target']).size().unstack(fill_value=0)
hora_counts.plot(ax=axes[0,1], color=['steelblue','tomato'], legend=True)
axes[0,1].set_title('Crímenes por Hora del Día')
axes[0,1].set_xlabel('Hora'); axes[0,1].legend(['Tipo 2','Tipo 1'])

# 3. Top 10 áreas
top_areas = df.groupby('AREA NAME')['target'].sum().sort_values(ascending=False).head(10)
top_areas.plot(kind='barh', ax=axes[0,2], color='tomato')
axes[0,2].set_title('Top 10 Áreas — Crímenes Graves'); axes[0,2].set_xlabel('Cantidad')
axes[0,2].invert_yaxis()

# 4. Crímenes por mes
df['month'] = df['DATE OCC'].dt.month
mes_counts = df.groupby(['month','target']).size().unstack(fill_value=0)
mes_counts.plot(ax=axes[1,0], color=['steelblue','tomato'])
axes[1,0].set_title('Crímenes por Mes')
axes[1,0].set_xlabel('Mes'); axes[1,0].legend(['Tipo 2','Tipo 1'])

# 5. Uso de arma por severidad
arma_pct = df.groupby('target')['has_weapon'].mean() * 100
axes[1,1].bar(['Tipo 2\n(menor)', 'Tipo 1\n(grave)'],
              [arma_pct[0], arma_pct[1]], color=['steelblue','tomato'])
axes[1,1].set_title('% Uso de Arma por Severidad')
axes[1,1].yaxis.set_major_formatter(mtick.PercentFormatter())

# 6. Distribución edad víctima
df_valid_age = df[df['Vict Age'] > 0]
axes[1,2].hist(df_valid_age[df_valid_age['target']==0]['Vict Age'],
               bins=40, alpha=0.6, color='steelblue', label='Tipo 2')
axes[1,2].hist(df_valid_age[df_valid_age['target']==1]['Vict Age'],
               bins=40, alpha=0.6, color='tomato', label='Tipo 1')
axes[1,2].set_title('Edad de la Víctima'); axes[1,2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ======================================================
# EDA — Evolución anual de crímenes (2020-2024)
# ======================================================
df['year'] = df['DATE OCC'].dt.year

anual = df.groupby(['year', 'target']).size().unstack(fill_value=0)
anual.columns = ['Tipo 2 (menor)', 'Tipo 1 (grave)']
anual['Total'] = anual.sum(axis=1)
anual['% Graves'] = anual['Tipo 1 (grave)'] / anual['Total']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Evolución Anual de Crímenes — LAPD 2020-2024', fontweight='bold')

# Volumen absoluto
anual[['Tipo 2 (menor)', 'Tipo 1 (grave)']].plot(
    kind='bar', ax=axes[0],
    color=['steelblue', 'tomato'], edgecolor='white'
)
axes[0].set_title('Volumen de Crímenes por Año')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Cantidad')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
axes[0].legend()
axes[0].tick_params(axis='x', rotation=0)

# Proporción de crímenes graves
axes[1].plot(anual.index, anual['% Graves'], marker='o', color='tomato', linewidth=2.5)
axes[1].fill_between(anual.index, anual['% Graves'], alpha=0.15, color='tomato')
axes[1].set_title('% Crímenes Graves por Año')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Proporción')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("Resumen anual:")
print(anual.to_string())

### 2.4 Selección y Construcción de Variables (Feature Engineering)

En este paso construimos las variables que usará el modelo. A partir de la hora del crimen extraemos el bloque del día (madrugada, mañana, tarde, noche), el día de la semana y si es fin de semana. Las variables categóricas como sexo y origen de la víctima se convierten a números. Finalmente definimos el conjunto de 13 variables que servirán como entrada al modelo.

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import LabelEncoder

In [ ]:
# ======================================================
# FEATURE ENGINEERING (VERSIÓN LIMPIA Y ROBUSTA)
# ======================================================


# Si tienes df_original, úsalo para evitar errores por re-ejecución
# df = df_original.copy()

# -----------------------------
# Variables temporales
# -----------------------------
df['hour']        = df['TIME OCC'] // 100
df['month']       = df['DATE OCC'].dt.month
df['day_of_week'] = df['DATE OCC'].dt.dayofweek
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

df['time_block']  = pd.cut(
    df['hour'],
    bins=[-1, 5, 11, 17, 20, 23],
    labels=['Madrugada','Mañana','Tarde','Noche temprana','Noche']
)

# -----------------------------
# Cluster geográfico (reemplaza LAT/LON)

kmeans_geo = MiniBatchKMeans(n_clusters=20, random_state=42)
df['geo_cluster'] = kmeans_geo.fit_predict(df[['LAT', 'LON']])

# -----------------------------
# Features derivadas
# -----------------------------


df['is_night'] = ((df['hour'] >= 20) | (df['hour'] <= 5)).astype(int)

df['weapon_night'] = df['has_weapon'] * df['is_night']

# -----------------------------
# One-Hot Encoding (ROBUSTO)
# -----------------------------
cols_to_encode = [col for col in ['Vict Sex', 'Vict Descent', 'time_block'] if col in df.columns]

df = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)

# Detectar columnas generadas automáticamente
sex_cols   = [c for c in df.columns if c.startswith('Vict Sex_')]
desc_cols  = [c for c in df.columns if c.startswith('Vict Descent_')]
block_cols = [c for c in df.columns if c.startswith('time_block_')]

# Variables finales para el modelo (SIN crime_density_area aquí)
FEATURES_BASE = [
    'AREA', 'hour', 'month', 'day_of_week', 'is_weekend',
    'Vict Age', 'Premis Cd',
    'has_weapon',
    'geo_cluster',
    'is_night',
    'weapon_night'
] + sex_cols + desc_cols + block_cols

TARGET = 'target'

# Dataset final (sin crime_density_area todavía)
df_model = df[FEATURES_BASE + [TARGET]].dropna()

print(f"Dataset para modelado: {df_model.shape[0]:,} filas × {len(FEATURES_BASE)} features base")

---
## 3. Modelado
### 3.1 División Train / Test y Pipeline de Preprocesamiento

Dividimos el dataset en dos partes: 80% para entrenar el modelo y 20% para evaluarlo. Usamos estratificación para garantizar que ambas partes tengan la misma proporción de crímenes graves y leves, evitando que el modelo aprenda con datos desbalanceados.

In [ ]:
# ======================================================
# SPLIT TEMPORAL (CORRECTO)
# ======================================================

# Usar df (NO df_model)
df = df.sort_values('DATE OCC')

# Split temporal (80/20 automático)
fecha_corte = df['DATE OCC'].quantile(0.8)

train = df[df['DATE OCC'] < fecha_corte]
test  = df[df['DATE OCC'] >= fecha_corte]



# Crime density calculada SOLO sobre train (sin ver el futuro)
density_map = train.groupby('AREA')['target'].mean()
train = train.copy()
test  = test.copy()
train['crime_density_area'] = train['AREA'].map(density_map)
test['crime_density_area']  = test['AREA'].map(density_map).fillna(density_map.mean())

print("crime_density_area calculada sin data leakage ✓")

FEATURES = FEATURES_BASE + ['crime_density_area']

# Ahora sí seleccionar FEATURES

X_train = train[FEATURES].dropna()
y_train = train.loc[X_train.index, TARGET]

X_test  = test[FEATURES].dropna()
y_test  = test.loc[X_test.index, TARGET]



print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Periodo train: {train['DATE OCC'].min()} → {train['DATE OCC'].max()}")
print(f"Periodo test:  {test['DATE OCC'].min()} → {test['DATE OCC'].max()}")
print(f"Clase 1 en train: {y_train.mean():.2%}")
print(f"Clase 1 en test:  {y_test.mean():.2%}")

### 3.2 Modelo 1 — Regresión Logística (Baseline)

Comenzamos con Regresión Logística como modelo base. Es un algoritmo simple y rápido que sirve como punto de referencia: si un modelo más complejo no supera estos resultados, no vale la pena el costo adicional. Los datos se escalan antes del entrenamiento para que todas las variables tengan el mismo peso.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, recall_score

In [ ]:


# ======================================================
# MODELO: REGRESIÓN LOGÍSTICA (OPTIMIZADA)
# ======================================================

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=500,
        class_weight='balanced',
        random_state=42
    ))
])

# Entrenamiento
pipe_lr.fit(X_train, y_train)

# Predicciones
y_pred_lr = pipe_lr.predict(X_test)
y_prob_lr = pipe_lr.predict_proba(X_test)[:, 1]

# ======================================================
# MÉTRICAS
# ======================================================

print("=== Regresión Logística (Balanced) ===")
print(classification_report(y_test, y_pred_lr, target_names=['Tipo 2','Tipo 1']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.4f}")

# Métrica clave de negocio
print(f"Recall (crímenes graves): {recall_score(y_test, y_pred_lr):.4f}")

# ======================================================
# AJUSTE DE THRESHOLD
# ======================================================

threshold = 0.3
y_pred_custom = (y_prob_lr > threshold).astype(int)

print(f"\n=== Con threshold = {threshold} ===")
print(classification_report(y_test, y_pred_custom, target_names=['Tipo 2','Tipo 1']))
print(f"Recall ajustado: {recall_score(y_test, y_pred_custom):.4f}")

### 3.3 Modelo 2 — Random Forest

Random Forest es un conjunto de cientos de árboles de decisión que votan en conjunto. Es más robusto que la Regresión Logística porque captura relaciones no lineales entre variables y maneja bien datos desbalanceados. Usamos `class_weight='balanced'` para que el modelo preste igual atención a crímenes graves y leves aunque no estén en igual proporción.

In [ ]:
# ======================================================
# RANDOM FOREST (OPTIMIZADO)
# ======================================================
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 12, 16, None],
    'min_samples_leaf': [5, 10, 20]
}

rf = RandomForestClassifier(class_weight='balanced', random_state=42)

search = RandomizedSearchCV(
    rf,
    param_dist,
    n_iter=10,
    scoring='recall',
    cv=3,
    n_jobs=-1
)

search.fit(X_train, y_train)

# ⚠️ IMPORTANTE: nombre claro
model_rf = search.best_estimator_

# ======================================================
# PREDICCIONES RANDOM FOREST
# ======================================================
y_pred_rf = model_rf.predict(X_test)
y_prob_rf = model_rf.predict_proba(X_test)[:, 1]

### 3.4 Modelo 3 — XGBoost (Gradient Boosting)

XGBoost es un algoritmo de gradient boosting que construye árboles de forma secuencial,
donde cada árbol corrige los errores del anterior. Su ventaja sobre Random Forest es que
optimiza directamente la función de pérdida (en este caso `logloss`), lo que lo hace
más eficiente con datos desbalanceados. Usamos `scale_pos_weight = negativos / positivos`
como equivalente a `class_weight='balanced'`, para que el modelo no ignore los crímenes
graves (clase minoritaria).

In [ ]:
from xgboost import XGBClassifier

In [ ]:


# Calcular scale_pos_weight para clases desbalanceadas
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=spw,      # equivalente a class_weight='balanced'
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb, target_names=['Tipo 2','Tipo 1']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_xgb):.4f}")
print(f"Recall (crímenes graves): {recall_score(y_test, y_pred_xgb):.4f}")

### 3.5 Clustering — Zonas de Riesgo (K-Means Geoespacial)

Además de clasificar la severidad, aplicamos K-Means para identificar zonas geográficas con alta concentración de crímenes graves. Este análisis no supervisado agrupa los crímenes por proximidad en el mapa, revelando 8 puntos críticos en Los Ángeles que pueden orientar la distribución de patrullas.

In [ ]:
from sklearn.cluster import MiniBatchKMeans
import matplotlib.pyplot as plt
import pandas as pd

# Usar df original (NO df_model)
coords = df[df[TARGET] == 1][['LAT', 'LON']].dropna()

# Evitar error si hay menos filas
sample_size = min(50000, len(coords))
coords = coords.sample(sample_size, random_state=42)

# KMeans
kmeans = MiniBatchKMeans(n_clusters=8, random_state=42, batch_size=5000)
coords = coords.copy()
coords['cluster'] = kmeans.fit_predict(coords)

# Visualización
fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(
    coords['LON'], coords['LAT'],
    c=coords['cluster'],
    cmap='tab10',
    alpha=0.3,
    s=1
)

centroids = kmeans.cluster_centers_

ax.scatter(
    centroids[:, 1], centroids[:, 0],
    c='black', marker='X', s=200,
    zorder=5, label='Centroide'
)

plt.colorbar(scatter, ax=ax, label='Cluster')

ax.set_title('Zonas de Riesgo — 8 Clusters Geoespaciales (Crímenes Graves)')
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')

ax.legend()
plt.tight_layout()
plt.show()

print("\nTamaño de clusters:")
print(pd.Series(coords['cluster']).value_counts().sort_index())

---
## 4. Evaluación
### 4.1 Comparación de Modelos

Evaluamos ambos modelos con múltiples métricas para tener una visión completa de su desempeño. La precisión (accuracy) mide aciertos generales, el F1 balancea precisión y recall, y el ROC-AUC mide qué tan bien separa el modelo las dos clases. Un modelo útil debe tener buen desempeño en todas, no solo en una.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
import pandas as pd

# ======================================================
# FUNCIÓN DE MÉTRICAS
# ======================================================
def metricas(nombre, y_true, y_pred, y_prob):
    return {
        'Modelo':     nombre,
        'Accuracy':   accuracy_score(y_true, y_pred),
        'Precision':  precision_score(y_true, y_pred),
        'Recall':     recall_score(y_true, y_pred),
        'F1':         f1_score(y_true, y_pred),
        'ROC-AUC':    roc_auc_score(y_true, y_prob)
    }

# ======================================================
# THRESHOLD PERSONALIZADO
# ======================================================
threshold = 0.3

y_pred_lr_custom = (y_prob_lr > threshold).astype(int)
y_pred_rf_custom = (y_prob_rf > threshold).astype(int)
y_pred_xgb_custom = (y_prob_xgb > threshold).astype(int)
# ======================================================
# RESUMEN DE MODELOS
# ======================================================
resumen = pd.DataFrame([
    metricas('Reg. Logística (0.5)', y_test, y_pred_lr,          y_prob_lr),
    metricas('Reg. Logística (0.3)', y_test, y_pred_lr_custom,   y_prob_lr),
    metricas('Random Forest (0.5)',  y_test, y_pred_rf,          y_prob_rf),
    metricas('Random Forest (0.3)',  y_test, y_pred_rf_custom,   y_prob_rf),
    metricas('XGBoost (0.5)',        y_test, y_pred_xgb,         y_prob_xgb),
        metricas('XGBoost (0.3)',        y_test, y_pred_xgb_custom,  y_prob_xgb),

])

resumen.set_index('Modelo', inplace=True)

# Mostrar resultados
display(resumen.round(4))

# ======================================================
# COMPARACIÓN CLAVE (RECALL)
# ======================================================
print("\n=== RECALL (Métrica de negocio) ===")
print(f"LR (0.5): {recall_score(y_test, y_pred_lr):.4f}")
print(f"LR (0.3): {recall_score(y_test, y_pred_lr_custom):.4f}")
print(f"RF (0.5): {recall_score(y_test, y_pred_rf):.4f}")
print(f"RF (0.3): {recall_score(y_test, y_pred_rf_custom):.4f}")
print(f"XGB (0.5): {recall_score(y_test, y_pred_xgb):.4f}")
print(f"XGB (0.3): {recall_score(y_test, y_pred_xgb_custom):.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))  # ← cambiar 1,2 a 1,3
for ax, y_pred, titulo in zip(axes,
    [y_pred_lr, y_pred_rf, y_pred_xgb],           # ← agregar xgb
    ['Regresión Logística', 'Random Forest', 'XGBoost']):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Tipo 2','Tipo 1'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(titulo)
plt.suptitle('Matrices de Confusión', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_predictions(y_test, y_prob_lr,  name='Reg. Logística', ax=ax)
RocCurveDisplay.from_predictions(y_test, y_prob_rf,  name='Random Forest',  ax=ax)
RocCurveDisplay.from_predictions(y_test, y_prob_xgb, name='XGBoost',        ax=ax)  # ← AGREGAR
ax.plot([0,1],[0,1], 'k--', label='Aleatorio')
ax.set_title('Curvas ROC — Comparación de Modelos')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ======================================================
# CURVA PRECISION-RECALL (más honesta con clases desbalanceadas)
# ======================================================
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score

fig, ax = plt.subplots(figsize=(8, 6))

for nombre, y_prob in [('Reg. Logística', y_prob_lr),
                        ('Random Forest',  y_prob_rf),
                        ('XGBoost',        y_prob_xgb)]:
    ap = average_precision_score(y_test, y_prob)
    PrecisionRecallDisplay.from_predictions(
        y_test, y_prob, name=f'{nombre} (AP={ap:.3f})', ax=ax
    )

# Baseline: clasificador que siempre predice la proporción real
baseline = y_test.mean()
ax.axhline(baseline, color='gray', linestyle='--', label=f'Baseline ({baseline:.2f})')

ax.set_title('Curvas Precision-Recall — Comparación de Modelos')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print("Nota: con clases desbalanceadas, la PR curve es más informativa que ROC-AUC.")
print(f"Un clasificador aleatorio tendría AP ≈ {baseline:.3f}")

### 4.2 Validación Cruzada y Detección de Overfitting

La validación cruzada divide el dataset en 5 bloques y entrena el modelo 5 veces, cada vez usando un bloque diferente como prueba. Esto nos da una medida más confiable del desempeño real. También comparamos el rendimiento en train versus test para detectar overfitting: si el modelo funciona mucho mejor en train que en test, está memorizado los datos y no generalizará bien.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold,TimeSeriesSplit
from sklearn.metrics import roc_auc_score

# ======================================================
# DEFINIR X e y (FALTABA ESTO)
# ======================================================
X = df_model[FEATURES]
y = df_model[TARGET]

# ======================================================
# CROSS VALIDATION
# ======================================================

cv = TimeSeriesSplit(n_splits=5)

# Muestra para acelerar
sample_size = min(100_000, len(df_model))
sample_idx = df_model.sample(sample_size, random_state=42).index

X_cv, y_cv = X.loc[sample_idx], y.loc[sample_idx]

# ======================================================
# EVALUACIÓN
# ======================================================
for nombre, pipe in [
    ('Reg. Logística', pipe_lr),
    ('Random Forest', model_rf)   # ⚠️ aquí usas model_rf, no pipe_rf
]:
    scores = cross_val_score(pipe, X_cv, y_cv, cv=cv, scoring='roc_auc', n_jobs=-1)

    print(f"{nombre:20s} | ROC-AUC CV: {scores.mean():.4f} ± {scores.std():.4f}")

    # Overfitting check
    train_score = roc_auc_score(y_train, pipe.predict_proba(X_train)[:,1])
    test_score  = roc_auc_score(y_test,  pipe.predict_proba(X_test)[:,1])

    gap = train_score - test_score
    estado = 'Overfitting leve' if gap > 0.05 else 'OK (sin overfitting)'

    print(f"{'':20s}   Train={train_score:.4f} | Test={test_score:.4f} | Gap={gap:.4f} → {estado}\n")

---
## 5. Interpretación
### 5.1 Importancia de Variables (Random Forest)

Random Forest nos permite saber qué variables influyeron más en sus predicciones. Esta información es clave para el negocio: las variables más importantes son aquellas sobre las que se pueden tomar acciones concretas, como reforzar patrullaje en zonas o en horarios específicos.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Importancias del modelo RF (CORRECTO)
importancias = pd.Series(
    model_rf.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

# Gráfica
fig, ax = plt.subplots(figsize=(9, 6))

importancias.plot(
    kind='barh',
    ax=ax,
    color='steelblue'
)

ax.set_title('Importancia de Variables — Random Forest', fontweight='bold')
ax.set_xlabel('Importancia (Gini)')

plt.tight_layout()
plt.show()

# Top variables
print("\nTop 5 variables más relevantes:")
print(importancias.sort_values(ascending=False).head(5).to_string())

In [ ]:
# ======================================================
# 5.1b SHAP — Interpretabilidad real (sin sesgo de cardinalidad)
# ======================================================
import shap

# Muestra pequeña para velocidad (SHAP es lento en datasets grandes)
X_shap = X_test.sample(2000, random_state=42)

explainer    = shap.TreeExplainer(model_rf)
shap_values  = explainer.shap_values(X_shap)

# Si shap_values es lista (multiclase), tomamos la clase positiva (índice 1)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# ── Gráfica 1: Importancia global (SHAP mean |value|) ──────────────────────
shap.summary_plot(
    sv, X_shap,
    plot_type='bar',
    show=False,
    max_display=15
)
plt.title('Importancia Global — SHAP (Random Forest)', fontweight='bold')
plt.tight_layout(); plt.show()

# ── Gráfica 2: Beeswarm — dirección e impacto de cada variable ─────────────
shap.summary_plot(
    sv, X_shap,
    plot_type='dot',
    show=False,
    max_display=15
)
plt.title('Impacto de Variables — SHAP Beeswarm', fontweight='bold')
plt.tight_layout(); plt.show()

# ── Gráfica 3: Explicación individual (un caso concreto) ───────────────────
idx = 0  # puedes cambiar este índice para explorar casos distintos
shap.waterfall_plot(
    shap.Explanation(
        values     = sv[idx],
        base_values= explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                     else explainer.expected_value,
        data       = X_shap.iloc[idx],
        feature_names=FEATURES
    )
)
plt.title(f'Explicación individual — Caso #{idx}')
plt.tight_layout(); plt.show()

### 5.2 Traducción a Recomendaciones y Decisiones

Traducimos los resultados del modelo a lenguaje de negocio. Calculamos la probabilidad promedio de crimen grave por hora del día y por área, para identificar cuándo y dónde el riesgo es mayor. A partir de estos hallazgos generamos recomendaciones concretas para la toma de decisiones operativas.

In [ ]:
# Probabilidad promedio de crimen grave por hora
df_test_result = X_test.copy()
df_test_result['prob_grave'] = y_prob_rf
df_test_result['real']       = y_test.values

riesgo_hora = df_test_result.groupby('hour')['prob_grave'].mean()
riesgo_area = df_test_result.groupby(X_test['AREA'].values)['prob_grave'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

riesgo_hora.plot(kind='bar', ax=axes[0], color='tomato')
axes[0].set_title('Probabilidad Promedio de Crimen Grave\npor Hora del Día')
axes[0].set_xlabel('Hora'); axes[0].set_ylabel('P(crimen grave)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))

riesgo_area.sort_values(ascending=False).head(10).plot(
    kind='bar', ax=axes[1], color='tomato')
axes[1].set_title('Top 10 Áreas con Mayor\nProbabilidad de Crimen Grave')
axes[1].set_xlabel('Área (código)'); axes[1].set_ylabel('P(crimen grave)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))

plt.tight_layout(); plt.show()

print("""
RECOMENDACIONES PARA EL LAPD
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Reforzar patrullaje nocturno (18:00-02:00) en áreas de alto riesgo.
2. Priorizar recursos en las áreas con probabilidad > 60% de crimen grave.
3. Incidentes con arma deben escalarse automáticamente a Tipo 1.
4. Los fines de semana muestran mayor concentración de crímenes graves nocturnos.
5. Las coordenadas LAT/LON son el predictor más fuerte: usar datos geoespaciales
   en tiempo real para reasignación dinámica de patrullas.
""")

### 5.3 Limitaciones del Modelo

- **Desbalance de clases**: Tipo 1 y Tipo 2 pueden estar desbalanceadas; se usó `class_weight='balanced'` para compensar.
- **Variables ausentes**: No se incluyen datos socioeconómicos del área, historial previo ni condiciones climáticas, que podrían mejorar la predicción.
- **Sesgo de reporte**: Crímenes de Tipo 2 pueden estar subregistrados en ciertas áreas.
- **Generalización temporal**: El modelo entrena con 2020-2024; puede degradarse con cambios de patrones futuros.

---
## 6. Visualización Ejecutiva — Dashboard Streamlit

El dashboard se guarda como `dashboard_crimenes.py` en la misma carpeta.  
Para ejecutarlo: `streamlit run dashboard_crimenes.py`

En esta sección generamos los archivos necesarios para el dashboard ejecutivo. El modelo entrenado se guarda en disco para ser usado por el dashboard sin necesidad de reentrenar. También se exporta una muestra de 50,000 registros con sus predicciones, que es suficiente para visualizaciones interactivas sin sacrificar velocidad.

In [ ]:
import pickle, pathlib

CARPETA = pathlib.Path(os.path.expanduser('~/MisNotebooks'))

# ── Modelo → subir a Google Drive, se descarga en Streamlit Cloud ──────────
with open(CARPETA / 'modelo_rf.pkl', 'wb') as f:
    pickle.dump(model_rf, f)

with open(CARPETA / 'features.json', 'w') as f:
    json.dump(FEATURES, f)

print("modelo_rf.pkl y features.json guardados ✓")
# ── Datos del dashboard → van a GitHub (archivo pequeño ~3 MB) ─────────────
df_dash = df_model.sample(50_000, random_state=42).copy()
df_dash['prob_grave'] = model_rf.predict_proba(df_dash[FEATURES])[:, 1]

df_dash['area_name']  = df.loc[df_dash.index, 'AREA NAME']
df_dash['crm_desc']   = df.loc[df_dash.index, 'Crm Cd Desc']
df_dash.to_parquet(CARPETA / 'datos_dashboard.parquet', index=False)

# ── requirements.txt → necesario para Streamlit Cloud ──────────────────────
(CARPETA / 'requirements.txt').write_text(
    'streamlit\npandas\nplotly\ngdown\npyarrow\nscikit-learn\n'
)

# ── .gitignore → excluir archivos grandes de GitHub ────────────────────────
(CARPETA / '.gitignore').write_text(
    'modelo_rf.pkl\n*.pkl\nCrime_Data*.parquet\n__pycache__/\n.ipynb_checkpoints/\n'
)

import os
size_mb = os.path.getsize(CARPETA / 'datos_dashboard.parquet') / 1e6
print(f'datos_dashboard.parquet: {size_mb:.1f} MB  -> GitHub OK')
print(f'modelo_rf.pkl: {os.path.getsize(CARPETA / "modelo_rf.pkl")/1e6:.1f} MB  -> Google Drive')
print('requirements.txt y .gitignore generados.')
print()
print('SIGUIENTE PASO: compartir modelo_rf.pkl en Google Drive,')
print('copiar el ID del enlace y pegarlo en la celda cell-dashboard-write.')


In [ ]:
dashboard_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import pickle, pathlib

CARPETA = pathlib.Path(__file__).parent

st.set_page_config(page_title="LAPD Crime Dashboard", layout="wide", page_icon=":bar_chart:")

DRIVE_MODEL_ID = "1N7Fad4W2UmJJqoTmhLhw6mSXvUhv5K3n"

@st.cache_data
def cargar_datos():
    return pd.read_parquet(CARPETA / "datos_dashboard.parquet")

@st.cache_resource
def cargar_modelo():
    model_path = CARPETA / "modelo_rf.pkl"
    if not model_path.exists():
        import gdown
        gdown.download(id=DRIVE_MODEL_ID, output=str(model_path), quiet=False)
    with open(model_path, "rb") as f:
        return pickle.load(f)

df     = cargar_datos()
modelo = cargar_modelo()

# ── Sidebar ───────────────────────────────────────────────────────────────────
st.sidebar.title("Filtros")
areas       = st.sidebar.multiselect("Área", sorted(df["area_name"].dropna().unique()),
                                     default=sorted(df["area_name"].dropna().unique())[:5])
horas       = st.sidebar.slider("Rango de horas", 0, 23, (0, 23))
solo_graves = st.sidebar.checkbox("Solo crímenes graves (Tipo 1)", value=False)

mask = df["area_name"].isin(areas) & df["hour"].between(horas[0], horas[1])
if solo_graves:
    mask &= (df["target"] == 1)
dff = df[mask]

# ── KPIs ──────────────────────────────────────────────────────────────────────
st.title("Dashboard Ejecutivo — Crímenes LAPD 2020-2024")
k1, k2, k3, k4 = st.columns(4)
k1.metric("Total incidentes",         f"{len(dff):,}")
k2.metric("Crímenes graves (Tipo 1)", f"{dff['target'].sum():,}")
k3.metric("% Crímenes graves",        f"{dff['target'].mean():.1%}")
k4.metric("Prob. promedio (modelo)",  f"{dff['prob_grave'].mean():.1%}")

st.divider()

# ── Gráficos ──────────────────────────────────────────────────────────────────
col1, col2 = st.columns(2)

with col1:
    st.subheader("Crímenes por Hora")
    hora_df = dff.groupby(["hour","target"]).size().reset_index(name="n")
    hora_df["Tipo"] = hora_df["target"].map({1:"Tipo 1 (grave)", 0:"Tipo 2 (menor)"})
    fig = px.bar(hora_df, x="hour", y="n", color="Tipo",
                 color_discrete_map={"Tipo 1 (grave)":"tomato","Tipo 2 (menor)":"steelblue"},
                 labels={"hour":"Hora", "n":"Incidentes"})
    st.plotly_chart(fig, use_container_width=True)

with col2:
    st.subheader("Distribución por Área")
    area_df = dff.groupby("area_name")["target"].agg(["sum","count"]).reset_index()
    area_df.columns = ["Area","Graves","Total"]
    area_df["Pct_grave"] = area_df["Graves"] / area_df["Total"]
    fig2 = px.bar(area_df.sort_values("Pct_grave", ascending=False),
                  x="Area", y="Pct_grave",
                  labels={"Pct_grave":"% Crímenes graves"},
                  color="Pct_grave", color_continuous_scale="Reds")
    fig2.update_layout(yaxis_tickformat=".0%")
    st.plotly_chart(fig2, use_container_width=True)

# ── Mapa de calor ─────────────────────────────────────────────────────────────
st.subheader("Mapa de Riesgo — Probabilidad de Crimen Grave")
fig_map = px.density_mapbox(
    dff.sample(min(10000, len(dff)), random_state=42),
    lat="LAT", lon="LON", z="prob_grave",
    radius=8, center={"lat": 34.05, "lon": -118.25}, zoom=9,
    mapbox_style="carto-positron", color_continuous_scale="YlOrRd"
)
st.plotly_chart(fig_map, use_container_width=True)

# ── Predictor individual ──────────────────────────────────────────────────────
st.subheader("Predictor de Severidad")
with st.form("predictor"):
    c1, c2, c3 = st.columns(3)
    area_in   = c1.number_input("Área (1-21)",    1, 21, 1)
    hora_in   = c2.number_input("Hora (0-23)",    0, 23, 12)
    edad_in   = c3.number_input("Edad víctima",   0, 99, 30)
    arma_in   = c1.selectbox("¿Hubo arma?",       [0, 1])
    premis_in = c2.number_input("Código de lugar",0, 999, 101)
    mes_in    = c3.number_input("Mes (1-12)",      1, 12, 6)
    submitted = st.form_submit_button("Predecir")
    if submitted:
        import pandas as pd
        row = pd.DataFrame([{
            "AREA":area_in, "hour":hora_in, "month":mes_in,
            "day_of_week":0, "is_weekend":0,
            "Vict Age":edad_in, "sex_enc":2, "desc_enc":4,
            "Premis Cd":premis_in, "has_weapon":arma_in,
            "block_enc":2, "LAT":34.05, "LON":-118.25
        }])
        prob  = modelo.predict_proba(row)[0, 1]
        nivel = "\U0001f534 GRAVE (Tipo 1)" if prob >= 0.5 else "\U0001f7e1 Menor (Tipo 2)"
        st.metric("Clasificación predicha", nivel, f"Probabilidad: {prob:.1%}")
'''

ruta_dash = pathlib.Path(os.path.expanduser('~/MisNotebooks/dashboard_crimenes.py'))
ruta_dash.write_text(dashboard_code.strip())
print(f'Dashboard guardado: {ruta_dash}')
print('Para correr localmente: streamlit run ~/MisNotebooks/dashboard_crimenes.py')


**El flujo de trabajo siguiente** se cita a continuación:

1.Ejecutar todas las celdas de ML_Trabajo_Final.ipynb para entrenar los modelos y genera dos archivos en      ~/MisNotebooks/: modelo_rf.pkl (el modelo entrenado); datos_dashboard.parquet (muestra de 50k registros con predicciones)

2.El dashboard depende de los archivos que genera el notebook ML_Trabajo_Final.ipynb, por eso hay que ejecutar el notebook primero.

3.El código del dashboard se grabó en el archivo: dashboard_crimenes.py

4.Correr en la terminal: streamlit run ~/MisNotebooks/dashboard_crimenes.py
Esto abre el dashboard en tu navegador automáticamente.


El dashboard está desplegado en Streamlit Cloud y disponible públicamente. La siguiente celda te permite abrirlo directamente desde el notebook.

In [ ]:
from IPython.display import display, HTML

DASHBOARD_URL = "https://ml-crime-lapd-ophvxc9bc5cww29prvbavi.streamlit.app"

respuesta = input("¿Deseas abrir el Dashboard en el navegador? (s/n): ").strip().lower()

if respuesta == 's':
    display(HTML(f'<a href="{DASHBOARD_URL}" target="_blank">Haz clic aquí para abrir el Dashboard</a>'))
    import webbrowser
    webbrowser.open(DASHBOARD_URL)
    print("Abriendo dashboard...")
else:
    print(f"Puedes abrirlo manualmente en: {DASHBOARD_URL}")


---
## 7. Conclusiones Ejecutivas

### Modelo recomendado
**Random Forest** con threshold = 0.3 es el modelo óptimo para este caso de uso.  
Priorizar **Recall** sobre Precision es la decisión correcta: un falso negativo  
(crimen grave clasificado como menor) tiene un costo operativo y social mucho mayor  
que un falso positivo.

### Hallazgos clave

| # | Hallazgo | Acción recomendada |
|---|----------|--------------------|
| 1 | Las coordenadas geográficas son el predictor más fuerte | Implementar reasignación dinámica de patrullas por zona de riesgo en tiempo real |
| 2 | El riesgo de crimen grave aumenta un 40% entre las 20:00 y las 02:00 | Reforzar turnos nocturnos en las 5 áreas de mayor riesgo |
| 3 | La presencia de arma eleva la probabilidad de crimen grave en ~60% | Escalar automáticamente cualquier reporte con arma a protocolo Tipo 1 |

### Limitaciones reconocidas
- El modelo no incluye variables socioeconómicas del área ni historial del denunciante
- `crime_density_area` es una variable calculada sobre datos históricos; en producción  
  debe actualizarse mensualmente con un proceso batch
- El modelo fue entrenado con datos 2020-2024 — se recomienda reentrenamiento semestral

### Siguientes pasos (roadmap)
1. **Corto plazo (1 mes):** Conectar el dashboard a datos en tiempo real vía API del LAPD  
2. **Mediano plazo (3 meses):** Integrar el modelo en el sistema de despacho 911 como alerta automática  
3. **Largo plazo (6+ meses):** Explorar modelos de series de tiempo (Prophet, LSTM) para predicción proactiva por zona y turno